In [40]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score , classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
import mlflow
from pathlib import Path


import matplotlib.pyplot as plt
import seaborn as sns

In [41]:
TRAIN_PATH = "./../data/processed/train.csv"
VALIDATION_PATH = "./../data/processed/validation.csv"
TEST_PATH = "./../data/processed/test.csv"

In [42]:
train_df= pd.read_csv(TRAIN_PATH)
validation_df= pd.read_csv(VALIDATION_PATH)
test_df= pd.read_csv(TEST_PATH)

X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]

X_validation = validation_df.drop(columns=["label"])
y_validation = validation_df["label"]

X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]

del train_df, validation_df, test_df

In [43]:
print("Training set shape:", X_train.shape)
print("Validation set shape:", X_validation.shape)
print("Test set shape:", X_test.shape)

Training set shape: (1623, 7)
Validation set shape: (541, 7)
Test set shape: (541, 7)


In [44]:
EXPERIMENT_NAME="multi_turn_jailbreak_experiment"
TRAIN_CONFUSION_MATRIX_PATH = './../reports/figures/train_confusion_matrix.png'
VALIDATION_CONFUSION_MATRIX_PATH = './../reports/figures/validation_confusion_matrix.png'
TEST_CONFUSION_MATRIX_PATH = './../reports/figures/test_confusion_matrix.png'
MLFLOW_DB_MODEL_PATH = './../models/mlflow.db'

In [45]:
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_MODEL_PATH}")
mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
experiment_id = experiment.experiment_id

print("EXPERIMENT INFO:")
print(f"Name: {experiment.name}")
print(f"ID: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"Tags: {experiment.tags}")
print(f"Lifecycle Stage: {experiment.lifecycle_stage}")
print(f"Creation timestamp: {experiment.creation_time}")

EXPERIMENT INFO:
Name: multi_turn_jailbreak_experiment
ID: 1
Artifact Location: file:///d:/FINAL_YEAR/gp/TCA/notebooks/mlruns/1
Tags: {}
Lifecycle Stage: active
Creation timestamp: 1780764988793


In [46]:
def save_confusion_matrix(cm, filename,title):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d") 
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.savefig(filename)

    plt.close()

In [47]:
def evaluate_model(y_pred, y_true):
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    return acc, report, cm

In [48]:
from evaluate import MultiTurnJailbreakEvaluator
import pandas as pd

evaluator = MultiTurnJailbreakEvaluator()

def group_by_conv(X, y_true, y_pred):
 
    df = X.copy()
    df["_label"] = y_true
    df["_pred"]  = y_pred

    # sort so turns are in order within each conversation
    df = df.sort_values(["conv_id"]).reset_index(drop=True)

    conv_ids       = []
    y_true_grouped = []
    y_pred_grouped = []

    for cid, group in df.groupby("conv_id", sort=False):
        conv_ids.append(cid)
        y_true_grouped.append(group["_label"].tolist())
        y_pred_grouped.append(group["_pred"].tolist())

    return conv_ids, y_true_grouped, y_pred_grouped


In [49]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(
    n_neighbors=min(10, len(X_train)-1),
    contamination=.2,
    metric="cosine"
)

y_pred_train = lof.fit_predict(X_train, y_train)
y_pred_train = [1 if p == -1 else 0 for p in y_pred_train]
acc_train, report_train, cm_train = evaluate_model(y_pred_train, y_train)



In [50]:
with mlflow.start_run(run_name='logistic_regression_v1', experiment_id=experiment_id):
    params = {
        'penalty': 'l2',         
        'C': 1.0,                
        'solver': 'lbfgs',       
        'max_iter': 10000,        
        'random_state': 42
    }

    logreg = LogisticRegression(**params)
    logreg.fit(X_train, y_train)

    y_train_pred = logreg.predict(X_train)
    y_validation_pred = logreg.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    print("Training Accuracy:", train_acc)
    print("Validation Accuracy:", val_acc)
    print("Training Classification Report:\n", train_report)
    print("Validation Classification Report:\n", val_report)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH+"logistic_regression_train_cm.png","train confusion matrix logistic regression")
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH+"logistic_regression_val_cm.png","validation confusion matrix logistic regression")

    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })
    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')
    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(logreg, 'logistic_regression_model')
    test_acc, test_report, test_cm = evaluate_model(logreg.predict(X_test), y_test)
    mlflow.log_metrics({'test_acc': test_acc})
    mlflow.log_text(test_report, 'test_classification_report.txt')
    save_confusion_matrix(test_cm, TEST_CONFUSION_MATRIX_PATH+"logistic_regression_test_cm.png","test confusion matrix logistic regression")
    mlflow.log_artifact(TEST_CONFUSION_MATRIX_PATH)

Training Accuracy: 0.6943930991990142
Validation Accuracy: 0.6894639556377079
Training Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.94      0.81      1085
           1       0.63      0.19      0.29       538

    accuracy                           0.69      1623
   macro avg       0.66      0.57      0.55      1623
weighted avg       0.68      0.69      0.64      1623

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.95      0.80       361
           1       0.62      0.17      0.27       180

    accuracy                           0.69       541
   macro avg       0.66      0.56      0.54       541
weighted avg       0.67      0.69      0.63       541



2026/06/08 02:25:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 02:25:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [51]:
with open("../reports/evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:logistic_regression \n" \
    "==============================\n" \
    "param_grid: {'C': 1.0, 'random_state': 42}\n" )
    # f.write("cv_best_score: " + str(grid_search.best_score_) + "\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_acc: " + str(test_acc) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
    

# # ── train ──────────────────────────────────────────────────────────
# conv_ids_train, y_true_train, y_pred_train = group_by_conv(
#     X_train, y_train, y_train_pred
# )
# result_train=evaluator.evaluate(y_true_train, y_pred_train )

# # ── validation ─────────────────────────────────────────────────────
# conv_ids_val, y_true_val, y_pred_val = group_by_conv(
#     X_validation, y_validation, y_validation_pred
# )
# result_val=evaluator.evaluate(y_true_val, y_pred_val, conv_ids=conv_ids_val)

In [52]:
with mlflow.start_run(run_name='logistic_regression_tuned_v1', experiment_id=experiment_id):

    base_model = LogisticRegression(max_iter=10000, random_state=42)

    param_grid = {
        "penalty": ["l2"],
        "C": [0.01, 0.1, 1, 10],
        "solver": ["lbfgs"]
    }

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    y_train_pred = best_model.predict(X_train)
    y_validation_pred = best_model.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    print("Best Params:", best_params)
    print("Training Accuracy:", train_acc)
    print("Validation Accuracy:", val_acc)
    print("Training Classification Report:\n", train_report)
    print("Validation Classification Report:\n", val_report)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH+"logistic_regression_train_cm.png", "train confusion matrix logistic regression")
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH+"logistic_regression_val_cm.png", "validation confusion matrix logistic regression")

    mlflow.log_params(best_params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc,
        'cv_best_score': grid_search.best_score_
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(best_model, 'logistic_regression_model')
    test_acc, test_report, test_cm = evaluate_model(best_model.predict(X_test), y_test)
    mlflow.log_metrics({'test_acc': test_acc})
    mlflow.log_text(test_report, 'test_classification_report.txt')
    save_confusion_matrix(test_cm, TEST_CONFUSION_MATRIX_PATH+"logistic_regression_test_cm.png", "test confusion matrix logistic regression")
    mlflow.log_artifact(TEST_CONFUSION_MATRIX_PATH)

Best Params: {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}
Training Accuracy: 0.6925446703635243
Validation Accuracy: 0.6876155268022182
Training Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.94      0.80      1085
           1       0.62      0.19      0.29       538

    accuracy                           0.69      1623
   macro avg       0.66      0.57      0.55      1623
weighted avg       0.67      0.69      0.63      1623

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.94      0.80       361
           1       0.60      0.18      0.27       180

    accuracy                           0.69       541
   macro avg       0.65      0.56      0.54       541
weighted avg       0.67      0.69      0.63       541



2026/06/08 02:25:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 02:25:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [53]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:logistic_regression \n" \
    "==============================\n" \
    "best_params: " + str(best_params) + "\n" )
    f.write("cv_best_score: " + str(grid_search.best_score_) + "\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )


In [54]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)

Training Accuracy: 0.6925446703635243
Validation Accuracy: 0.6876155268022182
Training Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.94      0.80      1085
           1       0.62      0.19      0.29       538

    accuracy                           0.69      1623
   macro avg       0.66      0.57      0.55      1623
weighted avg       0.67      0.69      0.63      1623

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.94      0.80       361
           1       0.60      0.18      0.27       180

    accuracy                           0.69       541
   macro avg       0.65      0.56      0.54       541
weighted avg       0.67      0.69      0.63       541



In [55]:
with mlflow.start_run(run_name='decision_tree_v1', experiment_id=experiment_id):
    params = {
        'max_depth': 5,           
        'min_samples_split': 5,    
        'min_samples_leaf': 10,     
        'max_features': None,      
        'random_state': 42
    }

    dtc = DecisionTreeClassifier(**params)
    dtc.fit(X_train, y_train)

    y_train_pred = dtc.predict(X_train)
    y_validation_pred = dtc.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH+"decision_tree_train_cm.png", "train confusion matrix decision tree")
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH+"decision_tree_val_cm.png", "validation confusion matrix decision tree")

    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(dtc, 'decision_tree_model')
    test_acc, test_report, test_cm = evaluate_model(dtc.predict(X_test), y_test)

2026/06/08 02:25:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 02:25:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [56]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)

Training Accuracy: 0.7818853974121996
Validation Accuracy: 0.7430683918669131
Training Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.88      0.84      1085
           1       0.70      0.59      0.64       538

    accuracy                           0.78      1623
   macro avg       0.76      0.73      0.74      1623
weighted avg       0.78      0.78      0.78      1623

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.84      0.81       361
           1       0.63      0.55      0.59       180

    accuracy                           0.74       541
   macro avg       0.71      0.69      0.70       541
weighted avg       0.74      0.74      0.74       541

Test Accuracy: 0.7375231053604436
Test Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.86      0.81       362
           1       0.63     

In [57]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:decision_tree \n" \
    "==============================\n" \
    "param_grid: {'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2 , 'random_state': 42 ,''}\n" )

    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )

# # ── train ──────────────────────────────────────────────────────────
# conv_ids_train, y_true_train, y_pred_train = group_by_conv(
#     X_train, y_train, y_train_pred
# )
# result_train=evaluator.evaluate(y_true_train, y_pred_train )

# # ── validation ─────────────────────────────────────────────────────
# conv_ids_val, y_true_val, y_pred_val = group_by_conv(
#     X_validation, y_validation, y_validation_pred
# )
# result_val=evaluator.evaluate(y_true_val, y_pred_val, conv_ids=conv_ids_val)

In [58]:
with mlflow.start_run(run_name='decision_tree_tuned_v1', experiment_id=experiment_id):

    base_model = DecisionTreeClassifier(random_state=42)

    param_grid = {
        "max_depth": [5, 10, 20, None],
        "min_samples_split": [2, 5],
        "min_samples_leaf":[5,10],

    }

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    y_train_pred = best_model.predict(X_train)
    y_validation_pred = best_model.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    print("Best Params:", best_params)
    print("Training Accuracy:", train_acc)
    print("Validation Accuracy:", val_acc)
    print("CV Best Score:", grid_search.best_score_)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH+"decision_tree_train_cm.png", "train confusion matrix decision tree")
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH+"decision_tree_val_cm.png", "validation confusion matrix decision tree")

    mlflow.log_params(best_params)

    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc,
        'cv_best_score': grid_search.best_score_
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(best_model, 'decision_tree_model')
    test_acc, test_report, test_cm = evaluate_model(best_model.predict(X_test), y_test)

Best Params: {'max_depth': 5, 'min_samples_leaf': 10, 'min_samples_split': 2}
Training Accuracy: 0.7818853974121996
Validation Accuracy: 0.7430683918669131
CV Best Score: 0.7049610819793457


2026/06/08 02:25:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 02:25:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [59]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)

Training Accuracy: 0.7818853974121996
Validation Accuracy: 0.7430683918669131
Training Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.88      0.84      1085
           1       0.70      0.59      0.64       538

    accuracy                           0.78      1623
   macro avg       0.76      0.73      0.74      1623
weighted avg       0.78      0.78      0.78      1623

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.84      0.81       361
           1       0.63      0.55      0.59       180

    accuracy                           0.74       541
   macro avg       0.71      0.69      0.70       541
weighted avg       0.74      0.74      0.74       541

Test Accuracy: 0.7375231053604436
Test Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.86      0.81       362
           1       0.63     

In [60]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:decision_tree_tuned_v1 \n" \
    "==============================\n" \
    "best_params: " + str(best_params) + "\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
# # # ── train ──────────────────────────────────────────────────────────
# conv_ids_train, y_true_train, y_pred_train = group_by_conv(
#     X_train, y_train, y_train_pred
# )
# result_train=evaluator.evaluate(y_true_train, y_pred_train )

# # ── validation ─────────────────────────────────────────────────────
# conv_ids_val, y_true_val, y_pred_val = group_by_conv(
#     X_validation, y_validation, y_validation_pred
# )
# result_val=evaluator.evaluate(y_true_val, y_pred_val, conv_ids=conv_ids_val)

In [61]:
with mlflow.start_run(run_name='random_forest_v1', experiment_id=experiment_id):
    params = {
    'n_estimators': 200,          
    'max_depth': 5,             
    'min_samples_split': 5,      
    'min_samples_leaf': 5,        
    'max_features': 'sqrt',       
    'bootstrap': True,
    'random_state': 42,
    'n_jobs': -1                 
    }

    rfc = RandomForestClassifier(**params)
    rfc.fit(X_train, y_train)
    y_train_pred = rfc.predict(X_train)
    y_validation_pred = rfc.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH+"random_forest_train_cm.png", "train confusion matrix random forest")
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH+"random_forest_val_cm.png", "validation confusion matrix random forest")

    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(rfc, 'random_forest_model')
    test_acc, test_report, test_cm = evaluate_model(rfc.predict(X_test), y_test)

2026/06/08 02:25:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 02:25:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [62]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)  
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)  

Training Accuracy: 0.7794208256315465
Validation Accuracy: 0.7634011090573013
Training Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.92      0.85      1085
           1       0.75      0.50      0.60       538

    accuracy                           0.78      1623
   macro avg       0.77      0.71      0.72      1623
weighted avg       0.78      0.78      0.77      1623

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.92      0.84       361
           1       0.74      0.45      0.56       180

    accuracy                           0.76       541
   macro avg       0.75      0.68      0.70       541
weighted avg       0.76      0.76      0.75       541

Test Accuracy: 0.7467652495378928
Test Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.91      0.83       362
           1       0.69     

In [63]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:random_forest_tuned_v1 \n" \
    "==============================\n" \
    "param_grid: {'n_estimators': 300,'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2,  'max_features': 'sqrt','bootstrap': True,'random_state': 42,'n_jobs': -1}'\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
# ── train ──────────────────────────────────────────────────────────
# conv_ids_train, y_true_train, y_pred_train = group_by_conv(
#     X_train, y_train, y_train_pred
# )
# result_train=evaluator.evaluate(y_true_train, y_pred_train )

# # ── validation ─────────────────────────────────────────────────────
# conv_ids_val, y_true_val, y_pred_val = group_by_conv(
#     X_validation, y_validation, y_validation_pred
# )
# result_val=evaluator.evaluate(y_true_val, y_pred_val, conv_ids=conv_ids_val)

In [64]:
with mlflow.start_run(run_name='random_forest_tuned_v1', experiment_id=experiment_id):

    base_model = RandomForestClassifier(random_state=42, n_jobs=-1)

    param_grid = {
        "n_estimators": [100, 300,400],
        "max_depth": [5,7,10, 20, None]
    }

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    y_train_pred = best_model.predict(X_train)
    y_validation_pred = best_model.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    print("Best Params:", best_params)
    print("Training Accuracy:", train_acc)
    print("Validation Accuracy:", val_acc)
    print("CV Best Score:", grid_search.best_score_)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH+"random_forest_tuned_train_cm.png", "train confusion matrix random forest")
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH+"random_forest_tuned_val_cm.png", "validation confusion matrix random forest")

    mlflow.log_params(best_params)

    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc,
        'cv_best_score': grid_search.best_score_
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(best_model, 'random_forest_model')
    test_acc, test_report, test_cm = evaluate_model(best_model.predict(X_test), y_test)

Best Params: {'max_depth': 5, 'n_estimators': 400}
Training Accuracy: 0.7868145409735059
Validation Accuracy: 0.7597042513863216
CV Best Score: 0.689372576850585


2026/06/08 02:26:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 02:26:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [65]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)  
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)

Training Accuracy: 0.7868145409735059
Validation Accuracy: 0.7597042513863216
Training Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.92      0.85      1085
           1       0.77      0.51      0.61       538

    accuracy                           0.79      1623
   macro avg       0.78      0.72      0.73      1623
weighted avg       0.78      0.79      0.77      1623

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.92      0.84       361
           1       0.73      0.44      0.55       180

    accuracy                           0.76       541
   macro avg       0.75      0.68      0.69       541
weighted avg       0.75      0.76      0.74       541

Test Accuracy: 0.7430683918669131
Test Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.91      0.83       362
           1       0.69     

In [66]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:random_forest_tuned_v1 \n" \
    "==============================\n" \
    "best_params: " + str(best_params) + "\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
# # ── train ──────────────────────────────────────────────────────────
# conv_ids_train, y_true_train, y_pred_train = group_by_conv(
#     X_train, y_train, y_train_pred
# )
# result_train=evaluator.evaluate(y_true_train, y_pred_train )

# # ── validation ─────────────────────────────────────────────────────
# conv_ids_val, y_true_val, y_pred_val = group_by_conv(
#     X_validation, y_validation, y_validation_pred
# )
# result_val=evaluator.evaluate(y_true_val, y_pred_val, conv_ids=conv_ids_val)

In [67]:
with mlflow.start_run(run_name='xgboost_v1', experiment_id=experiment_id):
    params = {
        'n_estimators': 300,        
        'max_depth': 3,             
        'learning_rate': 0.01,       
        'random_state': 42,
        'n_jobs': -1,
    }

    xgb = XGBClassifier(**params)
    xgb.fit(X_train, y_train)

    # Predictions
    y_train_pred = xgb.predict(X_train)
    y_validation_pred = xgb.predict(X_validation)

    # Evaluation
    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    # Save confusion matrices as images
    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH+"xgboost_train_cm.png", "train confusion matrix xgboost")
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH+"xgboost_val_cm.png", "validation confusion matrix xgboost")

    # Logging
    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(xgb, 'xgboost_model')
    test_acc, test_report, test_cm = evaluate_model(xgb.predict(X_test), y_test)

2026/06/08 02:26:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 02:26:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [68]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)

Training Accuracy: 0.7763401109057301
Validation Accuracy: 0.767097966728281
Training Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.91      0.85      1085
           1       0.74      0.50      0.60       538

    accuracy                           0.78      1623
   macro avg       0.76      0.71      0.72      1623
weighted avg       0.77      0.78      0.76      1623

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.91      0.84       361
           1       0.72      0.49      0.58       180

    accuracy                           0.77       541
   macro avg       0.75      0.70      0.71       541
weighted avg       0.76      0.77      0.75       541

Test Accuracy: 0.7430683918669131
Test Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.89      0.82       362
           1       0.67      

In [69]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:xgboost \n" \
    "==============================\n" \
    "param_grid: {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3 , 'random_state': 42}\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
    
# # ── train ──────────────────────────────────────────────────────────
# conv_ids_train, y_true_train, y_pred_train = group_by_conv(
#     X_train, y_train, y_train_pred
# )
# result_train=evaluator.evaluate(y_true_train, y_pred_train )

# # ── validation ─────────────────────────────────────────────────────
# conv_ids_val, y_true_val, y_pred_val = group_by_conv(
#     X_validation, y_validation, y_validation_pred
# )
# result_val=evaluator.evaluate(y_true_val, y_pred_val, conv_ids=conv_ids_val)

In [70]:
with mlflow.start_run(run_name='xgboost_tuned_v1', experiment_id=experiment_id):
    base_model = XGBClassifier(
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )

    param_grid = {
        "n_estimators": [200, 300,400,500],
        "max_depth": [3, 5,7,10],
        "learning_rate": [0.05, 0.01,0.1,0.04]
    }

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    y_train_pred = best_model.predict(X_train)
    y_validation_pred = best_model.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    print("Best Params:", best_params)
    print("Training Accuracy:", train_acc)
    print("Validation Accuracy:", val_acc)
    print("CV Best Score:", grid_search.best_score_)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH+"xgboost_tuned_train_cm.png", "train confusion matrix xgboost")
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH+"xgboost_tuned_val_cm.png", "validation confusion matrix xgboost")

    mlflow.log_params(best_params)

    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc,
        'cv_best_score': grid_search.best_score_
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(best_model, 'xgboost_model')
    test_acc, test_report, test_cm = evaluate_model(best_model.predict(X_test), y_test)


Fitting 5 folds for each of 64 candidates, totalling 320 fits
Best Params: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 400}
Training Accuracy: 0.7818853974121996
Validation Accuracy: 0.7615526802218114
CV Best Score: 0.7541443494776828


2026/06/08 02:26:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 02:26:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [71]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)


Training Accuracy: 0.7818853974121996
Validation Accuracy: 0.7615526802218114
Training Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.91      0.85      1085
           1       0.75      0.52      0.61       538

    accuracy                           0.78      1623
   macro avg       0.77      0.72      0.73      1623
weighted avg       0.78      0.78      0.77      1623

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.90      0.83       361
           1       0.71      0.48      0.57       180

    accuracy                           0.76       541
   macro avg       0.74      0.69      0.70       541
weighted avg       0.75      0.76      0.75       541

Test Accuracy: 0.7393715341959335
Test Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.90      0.82       362
           1       0.67     

In [72]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:xgboost_tuned_v1 \n" \
    "==============================\n" \
    "best_params: " + str(best_params) + "\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
# # # ── train ──────────────────────────────────────────────────────────
# conv_ids_train, y_true_train, y_pred_train = group_by_conv(
#     X_train, y_train, y_train_pred
# )
# result_train=evaluator.evaluate(y_true_train, y_pred_train )

# # ── validation ─────────────────────────────────────────────────────
# conv_ids_val, y_true_val, y_pred_val = group_by_conv(
#     X_validation, y_validation, y_validation_pred
# )
# result_val=evaluator.evaluate(y_true_val, y_pred_val, conv_ids=conv_ids_val)

In [73]:
with mlflow.start_run(run_name='svc_v1', experiment_id=experiment_id):
    params = {
        'C': 1.0,                
        'kernel': 'rbf',        
        'random_state': 42
    }

    svc = SVC(**params)
    svc.fit(X_train, y_train)

    y_train_pred = svc.predict(X_train)
    y_validation_pred = svc.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH+"svc_train_cm.png", "train confusion matrix svc")
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH+"svc_val_cm.png", "validation confusion matrix svc")

    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(svc, 'svc_model')
    test_acc, test_report, test_cm = evaluate_model(svc.predict(X_test), y_test)



2026/06/08 02:26:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 02:26:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [74]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)

Training Accuracy: 0.7048675292667899
Validation Accuracy: 0.6913123844731978
Training Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.87      0.80      1085
           1       0.59      0.37      0.45       538

    accuracy                           0.70      1623
   macro avg       0.66      0.62      0.63      1623
weighted avg       0.69      0.70      0.68      1623

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.87      0.79       361
           1       0.56      0.33      0.42       180

    accuracy                           0.69       541
   macro avg       0.64      0.60      0.60       541
weighted avg       0.67      0.69      0.67       541

Test Accuracy: 0.7042513863216266
Test Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.90      0.80       362
           1       0.60     

In [75]:


with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model: Support Vector Classifier\n" \
    "==============================\n" \
    "param_grid: {'C': 1.0, 'kernel': 'rbf', 'random_state': 42}\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
# ── train ──────────────────────────────────────────────────────────
# conv_ids_train, y_true_train, y_pred_train = group_by_conv(
#     X_train, y_train, y_train_pred
# )
# result_train=evaluator.evaluate(y_true_train, y_pred_train )

# # ── validation ─────────────────────────────────────────────────────
# conv_ids_val, y_true_val, y_pred_val = group_by_conv(
#     X_validation, y_validation, y_validation_pred
# )
# result_val=evaluator.evaluate(y_true_val, y_pred_val, conv_ids=conv_ids_val)

In [76]:
with mlflow.start_run(run_name='svc_tuned_v1', experiment_id=experiment_id):

    base_model = SVC()

    param_grid = {
        "C": [0.1, 1, 10,5],
        "kernel": ["rbf"],
        
    }

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    y_train_pred = best_model.predict(X_train)
    y_validation_pred = best_model.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    print("Best Params:", best_params)
    print("Training Accuracy:", train_acc)
    print("Validation Accuracy:", val_acc)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH+"svc_tuned_train_cm.png", "train confusion matrix svc")
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH+"svc_tuned_val_cm.png", "validation confusion matrix svc")

    mlflow.log_params(best_params)

    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc,
        'cv_best_score': grid_search.best_score_
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(best_model, 'svc_model')
    test_acc, test_report, test_cm = evaluate_model(best_model.predict(X_test), y_test)


Best Params: {'C': 5, 'kernel': 'rbf'}
Training Accuracy: 0.7091805298829328
Validation Accuracy: 0.7171903881700554


2026/06/08 02:26:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 02:26:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [77]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)

print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)

Training Accuracy: 0.7091805298829328
Validation Accuracy: 0.7171903881700554
Training Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.85      0.80      1085
           1       0.59      0.42      0.49       538

    accuracy                           0.71      1623
   macro avg       0.67      0.64      0.64      1623
weighted avg       0.69      0.71      0.69      1623

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.86      0.80       361
           1       0.61      0.43      0.50       180

    accuracy                           0.72       541
   macro avg       0.68      0.64      0.65       541
weighted avg       0.70      0.72      0.70       541

Test Accuracy: 0.7153419593345656
Test Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.88      0.80       362
           1       0.61     

In [78]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model: Support Vector Classifier\n" \
    "==============================\n" \
    "best_param  " + str(best_params) + "\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_acc: " + str(test_acc) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
# # ── train ──────────────────────────────────────────────────────────
# conv_ids_train, y_true_train, y_pred_train = group_by_conv(
#     X_train, y_train, y_train_pred
# )
# result_train=evaluator.evaluate(y_true_train, y_pred_train )

# # ── validation ─────────────────────────────────────────────────────
# conv_ids_val, y_true_val, y_pred_val = group_by_conv(
#     X_validation, y_validation, y_validation_pred
# )
# result_val=evaluator.evaluate(y_true_val, y_pred_val, conv_ids=conv_ids_val)